# Data Analysis for BSSE Project

This notebook contains Ryan's analysis of Ibrahim's data. 

## Reading in Data

The goal of this section is to read the data into data structures which are conducive to analysis. At the same time, we want to build in sanity checks to help us ensure that calculations ran successfully and that calculations are what we think they are. The end goal is to have the following data structures:

- Map from geometries to "short names"
- Table of energies with one row per "short name"

Misc. Notes.

- Data for the project lives in `bsse_db/data/monomer_name` where "monomer_name" is the molecular formula of the monomers in the cluster.
- We are going to assume that for a monomer containing $n$ atoms, the first $n$ atoms in a file belong to monomer 1, the next $n$ belong to monomer 2, the next $n$ belong to monomer 3, etc.
- Ghost atoms in NWChem are specified by prepending `bq` to the atomic symbol


In [21]:
import tarfile
import itertools
import os
import math
from nwchem_helpers.parse_nwchem_output import parse_nwchem_output

### Ne Clusters

- `Ne_dimers.tar` contain CCSD(T)/aug-cc-pvdz calculations (TODO: verify).
- `Ne_Ne_distance_x_y_z` directory contains a dimer where one of the monomers has been translated by $\vec{r} = (x,y,z)^T$.
- Translating like this duplicates effort because of the system's symmetry (i.e., only the distance matters)
- No 'output_E_B_B.txt' because monomers are the same.

In [30]:
nsteps    = 7    # The total number of displacements along each axis
step_size = 0.25 # How much we displace for each step.

distance_2_dimer_geom = {}

def to_point(atom):
    return [float(atom[i]) for i in range(1, 4)]

with tarfile.open('data/Ne/Ne_dimers.tar', 'r') as tarball:
    all_names = tarball.getnames() # Gets all the directories and files inside the tarball
    
    for dx, dy, dz in itertools.product(range(1, nsteps), range(1, nsteps), range(1, nsteps)):
        x, y, z = (dx * step_size, dy * step_size, dz * step_size)
        directory_name = os.path.join('Ne_dimers', 'Ne_Ne_distance_{}_{}_{}'.format(x, y, z))
    
        if directory_name in all_names: # Some points were skipped for being too close
            # Extract the results from the dimer file
            dimer_file_name = os.path.join(directory_name, 'output_E_AB_AB.txt')
            f = tarball.extractfile(dimer_file_name)
            content = f.read().decode("utf-8").split('\n')
            results = parse_nwchem_output(iter(content))

            # Map geometry to separation distance
            geom = results['Input Geometry (angstroms)']
            carts = [to_point(geom[i]) for i in range(2)]
            r = math.dist(carts[0], carts[1])
            if r in distance_2_dimer_geom:
                assert distance_2_dimer_geom[r] == geom
            else:
                distance_2_dimer_geom[r] = geom

                

32
